# 01b · Çok Detaylı EDA — Biohub Cell Tracking

`01_eda`'nın üstüne, **tüm dataset geneli** ve **metrik/linking/detection'a yönelik** derin analizler:

1. Ölçek & koordinat birimi tutarlılığı (tüm örnekler)
2. GT yoğunluğu: kare başına hücre, örnek başına soy
3. Soy analizi: uzunluk, süre, **zamansal boşluklar**
4. **Havuzlanmış hareket** (tüm örnekler): mesafe, eksen, hız → linking yarıçapı
5. **Hareket öngörülebilirliği** (yön kalıcılığı / hız otokorelasyonu)
6. **Kalabalıklık**: kare-içi en yakın komşu mesafesi vs 7 µm → *eşleşme belirsizliği*
7. **Bölünme geometrisi**: zaman, ebeveyn-kız & kız-kız mesafesi, simetri
8. Soy başlangıç/bitiş: alan sınırı mı, iç mi; zamanlama
9. **Detection** (görüntü örnekli): çekirdek yarıçapı, SNR, eşik, **blob sayımı → T_true / fazla-tahmin cezası**
10. Z davranışı: GT derinlik dağılımı
11. `sample_submission` formatı → grafımıza eşleme
12. Toplu bulgular → önerilen baseline hiperparametreleri

> Görseller inline + `/kaggle/working/figures/`'a kaydedilir.

## 0 · Kurulum + yardımcılar

In [ ]:
import importlib, subprocess, sys
def has(p):
    try: importlib.import_module(p); return True
    except Exception: return False
if not has("zarr"):
    subprocess.run([sys.executable,"-m","pip","install","-q","zarr"], check=False)
import os, warnings
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd, zarr
import matplotlib.pyplot as plt, matplotlib as mpl
from scipy.spatial import cKDTree
from scipy import ndimage as ndi
warnings.filterwarnings("ignore")
mpl.rcParams.update({"figure.facecolor":"white","axes.grid":True,"grid.alpha":0.25})
FIG = Path("/kaggle/working/figures"); FIG.mkdir(parents=True, exist_ok=True)
def savefig(n): plt.savefig(FIG/n, dpi=120, bbox_inches="tight"); plt.show()
SCALE_ZYX = (1.625, 0.40625, 0.40625)  # yedek; 1. bolumde dogrulanacak
print("hazir")

In [ ]:
def open_image(zpath):
    node = zarr.open(str(zpath), mode="r")
    attrs = dict(node.attrs) if hasattr(node,"attrs") else {}
    arr, scale = None, None
    ms = attrs.get("multiscales")
    if ms is None and isinstance(attrs.get("ome"), dict): ms = attrs["ome"].get("multiscales")
    if ms:
        try:
            ds0 = ms[0]["datasets"][0]; arr = node[ds0["path"]]
            for tf in ds0.get("coordinateTransformations", []):
                if tf.get("type")=="scale": scale = tuple(float(v) for v in tf["scale"][-3:])
        except Exception: pass
    if arr is None:
        keys = list(node.keys()) if hasattr(node,"keys") else []
        arr = node["0"] if "0" in keys else node
    return arr, attrs, scale

def load_geff_raw(gp):
    g = zarr.open(str(gp), mode="r"); ra = dict(g.attrs)
    axes = ra["axes"] if isinstance(ra.get("axes"), list) else None
    for k in ("geff","geff_metadata"):
        v = ra.get(k)
        if isinstance(v, dict) and v.get("axes"): axes = v["axes"]
    nodes = g["nodes"]; ids = np.asarray(nodes["ids"]); props={}
    if "props" in nodes:
        for pn in list(nodes["props"].keys()):
            try: props[pn]=np.asarray(nodes["props"][pn]["values"])
            except Exception: pass
    edges = np.asarray(g["edges"]["ids"])
    return {"node_ids":ids,"props":props,"edges":edges,"axes":axes}

def nodes_frame(gd):
    p=gd["props"]; axes=gd.get("axes") or {}
    tn=zn=yn=xn=None
    for a in (axes if isinstance(axes,list) else []):
        if not isinstance(a,dict): continue
        nm=a.get("name"); tp=str(a.get("type","")).lower()
        if tp in ("time","t") or nm in ("t","time","frame"): tn=nm
        elif nm in ("z","Z"): zn=nm
        elif nm in ("y","Y"): yn=nm
        elif nm in ("x","X"): xn=nm
    def pick(nm,*c):
        if nm and nm in p: return p[nm]
        for k in c:
            if k in p: return p[k]
        return None
    d={"id":gd["node_ids"]}
    for key,val in [("t",pick(tn,"t","time","frame","T")),("z",pick(zn,"z","Z")),
                    ("y",pick(yn,"y","Y")),("x",pick(xn,"x","X"))]:
        if val is not None: d[key]=val
    return pd.DataFrame(d)
print("yardimcilar hazir")

## 1 · Tüm grafları yükle + ölçek/koordinat tutarlılığı

In [ ]:
INPUT=Path("/kaggle/input")
def find_root():
    if not INPUT.exists(): return None
    st=[(INPUT,0)]
    while st:
        b,d=st.pop()
        try:
            if (b/"train").is_dir() and (b/"test").is_dir(): return b
        except Exception: pass
        if d<4:
            for c in sorted(b.iterdir()):
                if c.is_dir() and not c.name.endswith((".zarr",".geff")): st.append((c,d+1))
    return None
ROOT=find_root(); print("kok:",ROOT)
train_dir=ROOT/"train"; test_dir=ROOT/"test"
train_names=sorted(p.stem for p in train_dir.glob("*.zarr"))
test_names=sorted(p.stem for p in test_dir.glob("*.zarr"))
print("train:",len(train_names),"| test:",len(test_names))

In [ ]:
# Tum train .geff'leri yukle -> birlesik nodes + per-sample edges
GRAPHS={}   # name -> dict(ndf, edges)
for nm in train_names:
    try:
        gd=load_geff_raw(train_dir/(nm+".geff")); ndf=nodes_frame(gd)
        GRAPHS[nm]={"ndf":ndf,"edges":gd["edges"]}
    except Exception as e:
        print("[skip]",nm,e)
alln=pd.concat([g["ndf"].assign(sample=nm) for nm,g in GRAPHS.items()], ignore_index=True)
print("toplam node:",len(alln),"| ornek:",len(GRAPHS))
print("kolonlar:",list(alln.columns))
print(alln[["t","z","y","x"]].describe().round(2).to_string())

In [ ]:
# Goruntu boyut+olcek (metadata) tum orneklerde
srows=[]
for nm in train_names:
    try:
        arr,_,sc=open_image(train_dir/(nm+".zarr")); sh=arr.shape
        srows.append(dict(sample=nm,T=sh[0],Z=sh[1],Y=sh[2],X=sh[3],scale=sc,dtype=str(arr.dtype)))
    except Exception as e: print("[skip img]",nm,e)
shapes=pd.DataFrame(srows)
print("Boyutlar (benzersiz):")
print(shapes.groupby(["T","Z","Y","X"]).size().to_string())
scs=[s for s in shapes["scale"] if s]
print("\nOlcekler (benzersiz):", set(scs) if scs else "yok")
if scs: SCALE_ZYX=tuple(np.median(np.array(scs),axis=0)); print(">> SCALE_ZYX:",SCALE_ZYX)
print("dtype:", shapes["dtype"].unique())

In [ ]:
# Koordinat birimi: tum ornekler icin max koordinat vs boyut
dim=shapes.set_index("sample")[["Z","Y","X"]].to_dict("index")
rx=[]
for nm,g in GRAPHS.items():
    n=g["ndf"]
    if nm in dim and set(["x","y","z"]).issubset(n.columns):
        rx.append(n["x"].max()/dim[nm]["X"]);
allr=np.array(rx)
print("x_max/X orani -> medyan:%.3f  (voxelse ~1.0, um ise ~0.41)"%np.median(allr))
COORDS_UM = np.median(allr) < 0.6
print(">> COORDS_UM:",COORDS_UM,"(False=voxel)")
SZ,SY,SX=SCALE_ZYX
def to_um(dzyx):
    dzyx=np.asarray(dzyx,float)
    return dzyx if COORDS_UM else dzyx*np.array([SZ,SY,SX])

## 2 · GT yoğunluğu — kare başına hücre, örnek başına soy

In [ ]:
def uf(ids,edges):
    par={int(i):int(i) for i in ids}
    def f(a):
        while par[a]!=a: par[a]=par[par[a]]; a=par[a]
        return a
    for s,d in edges:
        s,d=int(s),int(d)
        if s in par and d in par: par[f(s)]=f(d)
    comp=defaultdict(list)
    for i in ids: comp[f(int(i))].append(int(i))
    return comp

per_sample=[]
nodes_per_frame=[]
for nm,g in GRAPHS.items():
    n=g["ndf"]; comp=uf(g["ndf"]["id"].values, g["edges"])
    outdeg=Counter(int(s) for s,_ in g["edges"])
    per_sample.append(dict(sample=nm,n_nodes=len(n),n_tracks=len(comp),
        n_div=sum(1 for v in outdeg.values() if v>=2),
        t_span=(int(n.t.max()-n.t.min()+1) if "t" in n else 0)))
    if "t" in n:
        nodes_per_frame += list(n.groupby("t").size().values)
ps=pd.DataFrame(per_sample); npf=np.array(nodes_per_frame)
print(ps[["n_nodes","n_tracks","n_div","t_span"]].describe().round(1).to_string())
print("\nkare basina node -> medyan:%.1f ort:%.1f max:%d"%(np.median(npf),npf.mean(),npf.max()))

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(17,4.5))
ax[0].hist(npf,bins=40,color="#4C78A8"); ax[0].set_title("kare başına etiketli node")
ax[0].set_xlabel("node/kare")
ax[1].hist(ps.n_tracks,bins=40,color="#F58518"); ax[1].set_title("örnek başına soy")
ax[1].set_xlabel("soy/örnek")
ax[2].scatter(ps.n_tracks,ps.n_nodes,s=12,alpha=0.6); ax[2].set_title("soy vs node")
ax[2].set_xlabel("soy"); ax[2].set_ylabel("node")
savefig("D02_gt_density.png")

## 3 · Soy analizi — uzunluk, süre, zamansal boşluklar

In [ ]:
lengths=[]; durations=[]; gap_flag=[]; div_in_comp=[]
for nm,g in GRAPHS.items():
    n=g["ndf"];
    if "t" not in n: continue
    id2t=dict(zip(n.id,n.t)); comp=uf(n["id"].values,g["edges"])
    outdeg=Counter(int(s) for s,_ in g["edges"])
    for root,members in comp.items():
        ts=[id2t[m] for m in members if m in id2t]
        if not ts: continue
        span=max(ts)-min(ts)+1; L=len(members)
        ndv=sum(1 for m in members if outdeg.get(m,0)>=2)
        lengths.append(L); durations.append(span); div_in_comp.append(ndv)
        # bolunmesiz soyda L<span => zamansal bosluk
        gap_flag.append(1 if (ndv==0 and L<span) else 0)
lengths=np.array(lengths); durations=np.array(durations); gap_flag=np.array(gap_flag)
print("soy sayisi:",len(lengths))
print("uzunluk (node) -> medyan:%.0f ort:%.1f max:%d"%(np.median(lengths),lengths.mean(),lengths.max()))
print("sure (kare) -> medyan:%.0f max:%d"%(np.median(durations),durations.max()))
print("bolunmesiz soylarda bosluk orani: %.1f%%"%(100*gap_flag.mean()))

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(17,4.5))
ax[0].hist(lengths,bins=50,color="#4C78A8"); ax[0].set_yscale("log")
ax[0].set_title("soy uzunluğu (node, log)"); ax[0].set_xlabel("node/soy")
ax[1].hist(durations,bins=50,color="#72B7B2"); ax[1].set_yscale("log")
ax[1].set_title("soy süresi (kare, log)"); ax[1].set_xlabel("kare")
frac_full=(lengths>=durations).mean()
ax[2].scatter(durations,lengths,s=8,alpha=0.4)
ax[2].plot([0,durations.max()],[0,durations.max()],"r--",label="boşluksuz")
ax[2].set_title("uzunluk vs süre"); ax[2].set_xlabel("süre"); ax[2].set_ylabel("uzunluk"); ax[2].legend()
savefig("D03_lineage.png")

## 4 · Havuzlanmış hareket (tüm örnekler) — linking yarıçapı

In [ ]:
disp=[]; DZ=[]; DY=[]; DX=[]; spt=[]
for nm,g in GRAPHS.items():
    n=g["ndf"]
    if not set(["z","y","x"]).issubset(n.columns): continue
    pos={int(i):p for i,p in zip(n.id.values, n[["z","y","x"]].values.astype(float))}
    id2t={int(i):tt for i,tt in zip(n.id.values, n.t.values)} if "t" in n else {}
    for s,d in g["edges"]:
        s,d=int(s),int(d); a=pos.get(s); b=pos.get(d)
        if a is not None and b is not None:
            dd=to_um(b-a)
            DZ.append(dd[0]);DY.append(dd[1]);DX.append(dd[2])
            disp.append(float(np.sqrt((dd**2).sum()))); spt.append(id2t.get(s,np.nan))
disp=np.array(disp)
for q in [50,90,95,99,100]:
    print("disp %d p: %.2f um"%(q,np.percentile(disp,q)))
print("edge sayisi:",len(disp))

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(17,4.5))
ax[0].hist(disp,bins=80,color="#4C78A8")
for q,c in [(95,"orange"),(99,"red")]:
    ax[0].axvline(np.percentile(disp,q),color=c,ls="--",label=f"{q}p")
ax[0].axvline(7,color="k",ls=":",label="7 µm")
ax[0].set_title("|yer değiştirme| (µm, tüm örnekler)"); ax[0].set_xlabel("µm"); ax[0].legend()
ax[1].hist(np.abs(DZ),bins=60,alpha=0.5,label="|dz|",density=True)
ax[1].hist(np.abs(DY),bins=60,alpha=0.5,label="|dy|",density=True)
ax[1].hist(np.abs(DX),bins=60,alpha=0.5,label="|dx|",density=True)
ax[1].set_title("eksen bazlı |Δ| (µm)"); ax[1].set_xlabel("µm"); ax[1].legend()
# kumulatif
xs=np.sort(disp); ax[2].plot(xs,np.arange(len(xs))/len(xs))
ax[2].axvline(7,color="k",ls=":"); ax[2].set_title("kümülatif dağılım")
ax[2].set_xlabel("µm"); ax[2].set_ylabel("oran")
savefig("D04_motion_pooled.png")
print(">> %.1f%% edge <7um"%(100*(disp<7).mean()))

## 5 · Hareket öngörülebilirliği — yön kalıcılığı

In [ ]:
# ardil adimlar arasi kosinus (tek girisli+tek cikisli node'larda)
cosv=[]; step_um=[]
for nm,g in GRAPHS.items():
    n=g["ndf"]
    if not set(["z","y","x"]).issubset(n.columns): continue
    pos={int(i):p for i,p in zip(n.id.values, n[["z","y","x"]].values.astype(float))}
    pred=defaultdict(list); succ=defaultdict(list)
    for s,d in g["edges"]:
        s,d=int(s),int(d); succ[s].append(d); pred[d].append(s)
    for nid in pos:
        if len(pred[nid])==1 and len(succ[nid])==1:
            p=pred[nid][0]; c=succ[nid][0]
            if p in pos and c in pos:
                v1=to_um(pos[nid]-pos[p]); v2=to_um(pos[c]-pos[nid])
                a=np.linalg.norm(v1)*np.linalg.norm(v2)
                if a>1e-6:
                    cosv.append(float(v1.dot(v2)/a)); step_um.append(float(np.linalg.norm(v2)))
cosv=np.array(cosv)
print("ardil adim kosinus -> medyan:%.2f ort:%.2f  (1=duz,0=rastgele,-1=geri)"%(np.median(cosv),cosv.mean()))
plt.figure(figsize=(11,4))
plt.subplot(1,2,1); plt.hist(cosv,bins=50,color="#4C78A8")
plt.title("ardıl adım yön kosinüsü"); plt.xlabel("cos θ")
plt.subplot(1,2,2); plt.scatter(step_um,cosv,s=5,alpha=0.3)
plt.title("adım uzunluğu vs yön kalıcılığı"); plt.xlabel("µm"); plt.ylabel("cos θ")
savefig("D05_persistence.png")

## 6 · Kalabalıklık — kare-içi en yakın komşu vs 7 µm (*eşleşme belirsizliği*)

In [ ]:
nn=[]
for nm,g in GRAPHS.items():
    n=g["ndf"]
    if not set(["t","z","y","x"]).issubset(n.columns): continue
    for t,grp in n.groupby("t"):
        if len(grp)<2: continue
        pts=grp[["z","y","x"]].values.astype(float)
        pts=pts if COORDS_UM else pts*np.array([SZ,SY,SX])
        tree=cKDTree(pts); dd,_=tree.query(pts,k=2); nn+=list(dd[:,1])
nn=np.array(nn)
if len(nn):
    print("kare-ici en yakin GT komsu (um) -> medyan:%.1f  5p:%.1f  <7um:%.1f%%"%(
        np.median(nn),np.percentile(nn,5),100*(nn<7).mean()))
    plt.figure(figsize=(11,4))
    plt.subplot(1,2,1); plt.hist(nn,bins=60,color="#E45756")
    plt.axvline(7,color="k",ls="--",label="7 µm"); plt.legend()
    plt.title("kare-içi en yakın GT komşu (µm)"); plt.xlabel("µm")
    plt.subplot(1,2,2); xs=np.sort(nn); plt.plot(xs,np.arange(len(xs))/len(xs))
    plt.axvline(7,color="k",ls="--"); plt.title("kümülatif"); plt.xlabel("µm")
    savefig("D06_crowding.png")
    print(">> Uyari: <7um komsular metrik eslesmesinde belirsizlik yaratabilir")
else:
    print("cogu karede <2 node (cok seyrek) — kalabaliklik olculemedi")

## 7 · Bölünme geometrisi — zaman, mesafeler, simetri

In [ ]:
dt=[]; pd_d=[]; dd_d=[]
for nm,g in GRAPHS.items():
    n=g["ndf"]
    if not set(["t","z","y","x"]).issubset(n.columns): continue
    P=n.set_index("id")[["z","y","x"]]; id2t=dict(zip(n.id,n.t)); idx=set(P.index)
    succ=defaultdict(list)
    for s,d in g["edges"]: succ[int(s)].append(int(d))
    for par,kids in succ.items():
        if len(kids)>=2 and par in idx:
            k=[c for c in kids if c in idx][:2]
            if len(k)<2: continue
            dt.append(id2t.get(par,np.nan))
            pd_d.append(float(np.linalg.norm(to_um(P.loc[k[0]].values-P.loc[par].values))))
            pd_d.append(float(np.linalg.norm(to_um(P.loc[k[1]].values-P.loc[par].values))))
            dd_d.append(float(np.linalg.norm(to_um(P.loc[k[0]].values-P.loc[k[1]].values))))
print("toplam bolunme:",len(dd_d))
if dd_d:
    print("kiz-kiz mesafe (um) -> medyan:%.1f  95p:%.1f"%(np.median(dd_d),np.percentile(dd_d,95)))
    print("ebeveyn-kiz mesafe (um) -> medyan:%.1f"%np.median(pd_d))
    fig,ax=plt.subplots(1,3,figsize=(17,4.5))
    ax[0].hist([x for x in dt if np.isfinite(x)],bins=30,color="#E45756"); ax[0].set_title("bölünme zamanı")
    ax[0].set_xlabel("t")
    ax[1].hist(dd_d,bins=30,color="#4C78A8"); ax[1].set_title("kız-kız mesafe (µm)"); ax[1].set_xlabel("µm")
    ax[2].hist(pd_d,bins=30,color="#F58518"); ax[2].set_title("ebeveyn-kız mesafe (µm)"); ax[2].set_xlabel("µm")
    savefig("D07_division.png")
else:
    print("veri kumesinde bolunme az/yok")

## 8 · Soy başlangıç/bitiş — alan sınırı mı, iç mi?

In [ ]:
starts=[]; ends=[]
for nm,g in GRAPHS.items():
    n=g["ndf"]
    if not set(["t","z","y","x"]).issubset(n.columns): continue
    indeg=Counter(int(d) for _,d in g["edges"]); outdeg=Counter(int(s) for s,_ in g["edges"])
    tmax=int(n.t.max())
    for _,r in n.iterrows():
        i=int(r.id)
        if indeg.get(i,0)==0: starts.append((r.t,r.z,r.y,r.x))
        if outdeg.get(i,0)==0: ends.append((r.t,r.z,r.y,r.x,tmax))
S=pd.DataFrame(starts,columns=["t","z","y","x"]); Ec=pd.DataFrame(ends,columns=["t","z","y","x","tmax"])
print("baslangic:",len(S),"| bitis:",len(Ec))
plt.figure(figsize=(16,4))
plt.subplot(1,3,1); plt.hist(S.t,bins=30,alpha=0.6,label="başlangıç"); plt.hist(Ec.t,bins=30,alpha=0.6,label="bitiş")
plt.legend(); plt.title("başlangıç/bitiş zamanı"); plt.xlabel("t")
plt.subplot(1,3,2); plt.scatter(S.x,S.y,s=5,alpha=0.4); plt.title("başlangıç konumu XY"); plt.xlabel("x"); plt.ylabel("y")
plt.subplot(1,3,3); plt.scatter(Ec.x,Ec.y,s=5,alpha=0.4,c="r"); plt.title("bitiş konumu XY"); plt.xlabel("x")
savefig("D08_endpoints.png")
# t=0'da baslamayan / son karede bitmeyen = gercek appear/disappear
print("t>0 baslangic (gercek beliriş): %.0f%%"%(100*(S.t>0).mean()))
print("t<tmax bitis (gercek kayboluş): %.0f%%"%(100*(Ec.t<Ec.tmax).mean()))

## 9 · Detection (görüntü örnekli) — çekirdek boyutu, SNR, eşik, blob sayımı

In [ ]:
# Birkac ornek x birkac zaman noktasi ile detection analizi
import random
NS=min(4,len(train_names)); NT=3
sel_names=[train_names[i] for i in np.linspace(0,len(train_names)-1,NS).astype(int)]
prof=np.zeros(25); prof_n=0; snr=[]; blob_counts=[]; gt_counts=[]; otsu_list=[]
for nm in sel_names:
    try:
        arr,_,_=open_image(train_dir/(nm+".zarr")); n=GRAPHS[nm]["ndf"]
        if "t" not in n: continue
        ts=sorted(set(n.t.astype(int)))
        pick=[ts[i] for i in np.linspace(0,len(ts)-1,min(NT,len(ts))).astype(int)]
        for t in pick:
            v=np.asarray(arr[t]).astype(np.float32); Z,Y,X=v.shape
            # Otsu esik
            hist,edges=np.histogram(v.ravel()[::5],bins=256)
            cdf=hist.cumsum(); tot=cdf[-1]; mids=(edges[:-1]+edges[1:])/2
            w1=cdf; w2=tot-cdf; mu1=np.cumsum(hist*mids); mu=mu1[-1]
            with np.errstate(invalid="ignore",divide="ignore"):
                between=(mu*w1-mu1)**2/(w1*w2+1e-9)
            thr=mids[np.nanargmax(between)]; otsu_list.append(float(thr))
            # blob sayimi: local max + esik
            mx=ndi.maximum_filter(v,size=(3,7,7))
            peaks=(v==mx)&(v>thr)
            blob_counts.append(int(peaks.sum())); gt_counts.append(int((n.t==t).sum()))
            # cekirdek radyal profil + SNR (GT cevresi)
            sub=n[n.t==t]
            for _,r in sub.iterrows():
                z,y,x=int(r.z),int(r.y),int(r.x)
                if not(0<=z<Z and 12<=y<Y-12 and 12<=x<X-12): continue
                patch=v[z, y-12:y+13, x-12:x+13]
                yy,xx=np.mgrid[-12:13,-12:13]; rad=np.sqrt(yy**2+xx**2).astype(int)
                for rr in range(25):
                    m=rad==rr
                    if m.any(): prof[rr]+=patch[m].mean()
                prof_n+=1
                bg=patch[rad>=10].mean(); ctr=v[z,y,x]
                if bg>0: snr.append(ctr/bg)
    except Exception as e: print("[skip det]",nm,e)
if prof_n: prof/=prof_n
print("kullanilan (ornek,zaman):",NS,"x",NT)
print("Otsu esik medyan:%.0f"%np.median(otsu_list))
if snr: print("SNR (merkez/halka) medyan:%.1f"%np.median(snr))
print("blob/kare medyan:%.0f | GT/kare medyan:%.0f"%(np.median(blob_counts),np.median(gt_counts)))

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(17,4.5))
rr=np.arange(25)*SX  # yaricap um (XY)
ax[0].plot(rr,prof,"-o",ms=3); half=prof.min()+(prof[0]-prof.min())/2
ax[0].axhline(half,color="r",ls="--",label="yarı-maks")
ax[0].set_title("çekirdek radyal yoğunluk profili"); ax[0].set_xlabel("yarıçap (µm)"); ax[0].legend()
if snr:
    ax[1].hist(snr,bins=40,color="#72B7B2"); ax[1].set_title("SNR (merkez/halka)"); ax[1].set_xlabel("oran")
ax[2].scatter(gt_counts,blob_counts,s=30);
mx=max(max(blob_counts),1); ax[2].set_title("blob/kare vs GT/kare")
ax[2].set_xlabel("GT (etiketli)"); ax[2].set_ylabel("tespit blob (tahmini gerçek hücre)")
savefig("D09_detection.png")
print(">> blob/kare, gercek hucre sayisinin (T_true) kabaca ustsiniri; fazla-tahmin cezasi icin referans")

## 10 · Z (derinlik) davranışı — GT dağılımı

In [ ]:
if "z" in alln:
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1); plt.hist(alln.z,bins=40,color="#4C78A8")
    plt.title("GT node z-dağılımı (voxel)"); plt.xlabel("z")
    plt.subplot(1,2,2);
    if "t" in alln:
        plt.hexbin(alln.t,alln.z,gridsize=30,cmap="viridis"); plt.colorbar(label="node")
        plt.title("z vs t (GT yoğunluğu)"); plt.xlabel("t"); plt.ylabel("z")
    savefig("D10_zbehavior.png")
    print("z aralik:",float(alln.z.min()),"-",float(alln.z.max()))

## 11 · `sample_submission.csv` formatı

In [ ]:
ss=ROOT/"sample_submission.csv"
if ss.exists():
    sdf=pd.read_csv(ss)
    print("shape:",sdf.shape,"| kolonlar:",list(sdf.columns))
    print(sdf.head(12).to_string(index=False)); print("\ndtypes:\n",sdf.dtypes)
    print("\nbenzersiz deger sayisi:\n", sdf.nunique())
else: print("yok")

## 12 · Toplu Bulgular → Baseline Hiperparametreleri (çalıştırınca doldur)

**Linking**
- disp 95p / 99p = **… / … µm** → `max_distance ≈ …`
- yön kalıcılığı medyan cos = **…** (yüksekse hareket modeli ekle)
- kare-içi en yakın komşu <7µm oranı = **…%** (yüksekse eşleşme belirsizliği riski)

**Detection**
- Otsu eşik ≈ **…**, SNR medyan = **…**, çekirdek yarıçapı ≈ **… µm**
- blob/kare ≈ **…** → T_true tahmini; fazla-tahmin cezası için `T_pred ≈ …` hedefle

**Bölünme** (%10 ağırlık)
- toplam bölünme = **…**, kız-kız mesafe medyan = **… µm**

**Soy**
- medyan uzunluk/süre = **…**, boşluk oranı = **…%** (yüksekse gap-closing gerek)

**Sonraki:** `notebooks/02_baseline_ultrack.ipynb`.